# BT4012 Kaggle competition: detecting illicit Bitcoin transactions

**Student Name:** Lai Hsuan Jung
**Student ID:** E0968110

This notebook reproduces my final submission from the provided data. It covers the data, the
validation scheme, both models, the blend and the submission file. The experiments that led to these
choices are in `notebooks/01_eda.ipynb` to `notebooks/16_gnn_seeds_blend.ipynb`, and every experiment
and submission is listed in `experiment_log.md`.

## Final approach in one paragraph

Each transaction is scored by two models, and the two rankings are averaged:

1. **LightGBM** (gradient-boosted trees, library default settings) on the 165 raw transaction features,
   trained on the labeled transactions only.
2. **A graph neural network (GraphSAGE)** that combines each transaction's features with those of its
   predecessors and successors in the payment graph. Five copies are trained with different random
   seeds and their predictions are averaged.

The final score is `0.6 x LightGBM rank + 0.4 x GNN rank`.

## Results

| Submission | Model | Validation (fold mean) | Public leaderboard |
|---|---|---|---|
| `sub01_baseline` | LightGBM alone | 0.9808 | 0.94660 |
| `sub10_gnn5_blend_w04` | **this notebook:** LightGBM + 0.4 x GNN | 0.9770 | **0.95654** |

The course baseline on the public leaderboard is 0.92119. My two selected final submissions are
`sub10_gnn5_blend_w04` and `sub01_baseline` (section 8 explains why).

## How to run

1. Create the environment: `python3 -m venv .venv && .venv/bin/pip install -r requirements.txt`
2. Put `train.csv`, `test.csv`, `txs_edgelist.csv` and `sample_submission.csv` in `data/`.
3. Open this notebook from the repository root and run all cells. It takes about 20 minutes, almost
   all of it training the five GNNs.

**Why part of the code runs as a separate script:** On my Mac, PyTorch (used for the GNN) and LightGBM
cannot train in the same Python session: the kernel crashes or freezes, because each library loads its
own copy of the OpenMP library it uses for multi-threading. So section 5 writes the GNN code to
`gnn_step.py` and runs it as a separate Python process, which saves its predictions to `outputs/`.
The rest of the notebook (LightGBM, blending, submission) runs here as usual. PyTorch is never
imported in this notebook.

## 1. Setup and data

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42             # used for LightGBM; the GNN script sets its own seeds (42 to 46)
DATA_DIR = "data"     # paths are relative to the repository root
OUT_DIR = "outputs"   # predictions written by the GNN script
SUB_DIR = "submissions"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SUB_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

# 165 anonymised features are columns named feat_1 ... feat_165
raw_cols = [c for c in train.columns if c.startswith("feat_")]

print("train:", train.shape, " test:", test.shape, " edges:", edges.shape)

train: (141772, 168)  test: (15329, 168)  edges: (234355, 2)


### What the data looks like

- Each row is one transaction, with an ID (`txId`), a time step and 165 numeric features. There are no
  missing values.
- `label` is 1 for illicit, 0 for licit, and missing for unlabeled transactions. Only labeled rows can
  be used to train and score a model.
- Training data covers time steps 1 to 35 and the test data steps 36 to 49, so the task is to predict
  the future.
- Each row of the edge list `(txId1, txId2)` means an output of `txId1` was spent by `txId2`. Edges never
  connect two different time steps. Some edges touch transactions that are in neither file; those have
  no features.

In [4]:
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

print("Training rows:", len(train), " labeled:", len(labeled),
      f"({len(labeled) / len(train):.0%})")
print("Illicit share among labeled rows:", round(labeled["label"].mean(), 4))
print("Train time steps:", train["time_step"].min(), "to", train["time_step"].max(),
      "| test time steps:", test["time_step"].min(), "to", test["time_step"].max())

Training rows: 141772  labeled: 31235 (22%)
Illicit share among labeled rows: 0.1167
Train time steps: 1 to 35 | test time steps: 36 to 49


## 2. Validation: three expanding-window folds

Because the test set lies in the future, every check trains on earlier time steps and scores later
ones. Random cross-validation would mix future rows into training and overstate the score.

| Fold | Train on steps | Validate on steps |
|---|---|---|
| 1 | 1 to 14 | 15 to 21 |
| 2 | 1 to 21 | 22 to 28 |
| 3 | 1 to 28 | 29 to 35 |

The score of a model is the ROC-AUC on each fold and the mean over the three. My first validation
scheme (one split: train 1 to 21, validate 22 to 35) turned out to be noisy and dominated by a few
time steps, so I changed to these folds (`notebooks/07_validation_scheme.ipynb`).

**Fold 2 is special.** Around step 24 a new kind of illicit transaction appears: groups of illicit
transactions connected to each other, which are rare in steps 1 to 21 (`notebooks/05_weak_steps.ipynb`).
Every model scores much lower on step 24, so fold 2 shows how a model copes with new behaviour.

In [5]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]

def split_fold(df, fold):
    # Return the (training rows, validation rows) of df for one fold
    tr = df[df["time_step"] <= fold["train_last"]]
    va = df[(df["time_step"] > fold["train_last"]) & (df["time_step"] <= fold["valid_last"])]
    return tr, va

for fold in FOLDS:
    tr, va = split_fold(labeled, fold)
    print(fold["name"], "| train rows:", len(tr), "illicit:", tr["label"].sum(),
          "| validation rows:", len(va), "illicit:", va["label"].sum())

fold 1 | train rows: 14896 illicit: 1005 | validation rows: 4634 illicit: 866
fold 2 | train rows: 19530 illicit: 1871 | validation rows: 5677 illicit: 671
fold 3 | train rows: 25207 illicit: 2542 | validation rows: 6028 illicit: 1102


## 3. How I arrived at this model

All results below are from my experiment notebooks and `experiment_log.md`. "Fold mean" uses the folds
above; the public leaderboard is Kaggle's score on 30% of the test set.

| What I tried | Result | Notebook |
|---|---|---|
| LightGBM, default settings, raw features | fold mean 0.9808, public 0.94660; the reference to beat | 02, 07 |
| Degree features (number of incoming/outgoing edges) | no gain on the folds (mean 0.9761) or the leaderboard (0.94770) | 03, 07 |
| Average of neighbours' features | no gain (mean 0.9770, public 0.94584) | 06, 07 |
| Dropping six features that drift strongly over time | worse (mean 0.9777, public 0.94365) | 08 |
| Ensemble with XGBoost and Random Forest | worse, especially on step 24 (mean 0.9723, public 0.94081) | 09 |
| Tuning LightGBM one setting at a time | no setting beat the defaults | 10 |
| Class weighting in LightGBM (`scale_pos_weight` 3 and full balance) | worse (mean 0.9768 and 0.9750) | 17 |
| Smoothing scores over the graph; weighting recent steps | no gain | 11, 12 |
| GraphSAGE GNN alone | weaker than LightGBM (mean 0.9478) | 13 |
| **LightGBM + GNN blend** | **public score rose from 0.94660 to 0.95654** | 14, 16 |
| Average of 5 GNN seeds instead of 1 | better GNN (mean 0.9531 against 0.9478) and better blend | 15, 16 |

**Imbalance.** About 11.7% of labeled rows are illicit.
- The evaluation metric, ROC-AUC, only depends on how the scores are ranked, so it is not inflated by
  the large licit class the way accuracy would be.
- LightGBM is trained without class weights. I tested weighting illicit rows 3 times as much, and by each
  fold's full licit-to-illicit ratio (about 9 to 14): both lowered the fold mean (0.9768 and 0.9750
  against 0.9808), almost entirely through fold 2 (`notebooks/17_class_weight.ipynb`).
- The GNN's loss weights illicit rows by the ratio of licit to illicit rows, which helps a neural
  network keep learning about the rare class.

**Overfitting:** LightGBM fits the training rows almost perfectly (training AUC about 1.0), but every
setting I tried that makes it simpler lowered fold 2, so the defaults were kept.

## 4. Model 1: LightGBM

LightGBM with library default settings (100 trees, learning rate 0.1, 31 leaves) on the 165 raw features.
It is trained on each fold here; the final model is trained in section 7.

In [ ]:
def to_rank(scores):
    # Position of each score (1 = lowest) divided by the count, so ranks run from 0 to 1.
    # ROC-AUC depends only on this order, and ranks put both models on the same scale.
    return pd.Series(scores).rank().values / len(scores)

lgbm_fold_scores = {}   
for fold in FOLDS:
    tr, va = split_fold(labeled, fold)
    model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    model.fit(tr[raw_cols], tr["label"])
    lgbm_fold_scores[fold["name"]] = model.predict_proba(va[raw_cols])[:, 1]   # column 1 = P(illicit)
    print(fold["name"], "LightGBM AUC:", round(roc_auc_score(va["label"], lgbm_fold_scores[fold["name"]]), 4))

fold 1 LightGBM AUC: 0.9888
fold 2 LightGBM AUC: 0.9572
fold 3 LightGBM AUC: 0.9965


## 5. Model 2: the graph neural network (separate script)

**What the GNN does:** Every transaction starts with its 165 (scaled) features. A GraphSAGE layer
computes a new vector for each transaction from its own vector and the average of its neighbours'
vectors, each multiplied by learned weights. Each of my layers has two parts that are added together:
one that looks at **predecessors** (transactions whose outputs it spends) and one that looks at
**successors** (transactions that spend its outputs), because money coming in and going out can mean
different things. Two layers are stacked, so each transaction sees its neighbours and their
neighbours. A final linear layer turns the vector into one score.

**Training:** All train and test transactions are nodes in one graph, so information flows through
unlabeled and test transactions too, but only the labeled rows of the training period count towards the
loss. No validation or test labels are used. Transactions outside train and test have no features and
are left out, with their edges. Features are scaled with the mean and standard deviation of the
training-period rows only.

| Setting | Value |
|---|---|
| layers / hidden size | 2 / 64 |
| dropout | 0.3 |
| optimiser | Adam, learning rate 0.01, weight decay 0.0005 |
| epochs | 100 (validation AUC stopped improving after about 25) |
| class imbalance | loss weight on illicit rows = licit rows / illicit rows |
| seeds | 42, 43, 44, 45, 46; the five predictions are averaged |

Averaging five seeds matters: a single GNN's fold score moved by up to 0.012 depending on its seed,
and the average beat every single seed (`notebooks/15_gnn_seeds.ipynb`).

**How the code runs:** The next cell does not run the GNN; `%%writefile` saves the cell's contents to
`gnn_step.py`. The cell after it runs that file as a separate Python process (`!` runs a terminal
command; `sys.executable` is this notebook's own Python, so the same environment is used). The script
writes the fold predictions and the test predictions to `outputs/`.

In [ ]:
%%writefile gnn_step.py
# GNN step of e0968110.ipynb. Run from the repository root: python gnn_step.py
# Trains 5 directed GraphSAGE models (seeds 42-46) on each validation fold and on all labeled rows,
# Saves the averaged predictions to outputs/.

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

DATA_DIR = "data"
OUT_DIR = "outputs"
EPOCHS = 100
HIDDEN = 64
SEEDS = [42, 43, 44, 45, 46]
FOLDS = [("fold 1", 14, 21), ("fold 2", 21, 28), ("fold 3", 28, 35)]   # (name, last train step, last valid step)
FINAL_TRAIN_LAST = 35

os.makedirs(OUT_DIR, exist_ok=True)
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

# One table of nodes: every train and test transaction 
nodes = pd.concat([train[["txId", "time_step", "label"] + raw_cols],
                   test[["txId", "time_step"] + raw_cols]], ignore_index=True)
row_of = pd.Series(np.arange(len(nodes)), index=nodes["txId"])   

# Keep edges whose two ends both have features. edge_index = [source rows, destination rows]
known = edges["txId1"].isin(row_of.index) & edges["txId2"].isin(row_of.index)
kept = edges[known]
edge_index = torch.tensor(np.vstack([row_of[kept["txId1"]].values, row_of[kept["txId2"]].values]),
                          dtype=torch.long)
time_step = nodes["time_step"].values
label = nodes["label"].values


class DirectedSAGE(torch.nn.Module):
    # Two GraphSAGE layers: each adds a predecessor part and a successor part.

    def __init__(self, n_features, hidden):
        super().__init__()
        self.pred1 = SAGEConv(n_features, hidden)
        self.succ1 = SAGEConv(n_features, hidden)
        self.pred2 = SAGEConv(hidden, hidden)
        self.succ2 = SAGEConv(hidden, hidden)
        self.out = torch.nn.Linear(hidden, 1)

    def forward(self, x, edge_index):
        # SAGEConv passes information from source to destination, so edge_index brings each node
        # its predecessors; flipping the two rows reverses every edge and brings its successors.
        reversed_edges = edge_index.flip(0)
        h = F.relu(self.pred1(x, edge_index) + self.succ1(x, reversed_edges))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.pred2(h, edge_index) + self.succ2(h, reversed_edges))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.out(h).squeeze(-1)    


def scaled_features(train_last_step):
    # Scale with the training-period mean and std, clip to -10..10
    in_train_period = time_step <= train_last_step
    mean = nodes.loc[in_train_period, raw_cols].mean()
    std = nodes.loc[in_train_period, raw_cols].std().replace(0, 1)
    return torch.tensor(((nodes[raw_cols] - mean) / std).clip(-10, 10).values, dtype=torch.float32)


def train_gnn(x, train_last_step, seed):
    # Train one GNN on the labeled rows up to train_last_step. Return a probability for every node
    train_rows = np.where((time_step <= train_last_step) & ~np.isnan(label))[0]
    y = torch.tensor(label[train_rows], dtype=torch.float32)
    pos_weight = (y == 0).sum() / (y == 1).sum()      

    torch.manual_seed(seed)                           # fix the starting weights and dropout
    model = DirectedSAGE(len(raw_cols), HIDDEN)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edge_index)                 
        loss = F.binary_cross_entropy_with_logits(logits[train_rows], y, pos_weight=pos_weight)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x, edge_index)).numpy()


def averaged_gnn(train_last_step, description):
    # Train one GNN per seed and average their probabilities
    x = scaled_features(train_last_step)
    probs = []
    for seed in SEEDS:
        probs.append(train_gnn(x, train_last_step, seed))
        print(f"  {description}: seed {seed} done", flush=True)
    return np.mean(probs, axis=0)


# 1) Validation folds: averaged predictions for each fold's labeled validation rows
fold_tables = []
for name, train_last, valid_last in FOLDS:
    avg = averaged_gnn(train_last, name)
    valid = (time_step > train_last) & (time_step <= valid_last) & ~np.isnan(label)
    fold_tables.append(pd.DataFrame({"fold": name, "txId": nodes["txId"].values[valid],
                                     "gnn_score": avg[valid]}))
pd.concat(fold_tables).to_csv(f"{OUT_DIR}/final_gnn_fold_predictions.csv", index=False)

# 2) Final model on all labeled rows (steps 1-35): averaged predictions for the test transactions
avg = averaged_gnn(FINAL_TRAIN_LAST, "final model")
is_test = np.isin(nodes["txId"].values, test["txId"].values)
pd.DataFrame({"txId": nodes["txId"].values[is_test], "gnn_score": avg[is_test]}) \
    .to_csv(f"{OUT_DIR}/final_gnn_test_predictions.csv", index=False)
print("GNN step finished: predictions saved to", OUT_DIR, flush=True)

Writing gnn_step.py


In [ ]:
# Run the GNN script in its own Python process 
!{sys.executable} gnn_step.py

  fold 1: seed 42 done
  fold 1: seed 43 done
  fold 1: seed 44 done
  fold 1: seed 45 done
  fold 1: seed 46 done
  fold 2: seed 42 done
  fold 2: seed 43 done
  fold 2: seed 44 done
  fold 2: seed 45 done
  fold 2: seed 46 done
  fold 3: seed 42 done
  fold 3: seed 43 done
  fold 3: seed 44 done
  fold 3: seed 45 done
  fold 3: seed 46 done
  final model: seed 42 done
  final model: seed 43 done
  final model: seed 44 done
  final model: seed 45 done
  final model: seed 46 done
GNN step finished: predictions saved to outputs


In [ ]:
# Load what the script saved
gnn_fold = pd.read_csv(f"{OUT_DIR}/final_gnn_fold_predictions.csv")
gnn_test = pd.read_csv(f"{OUT_DIR}/final_gnn_test_predictions.csv")

gnn_fold_scores = {}   
for fold in FOLDS:
    tr, va = split_fold(labeled, fold)
    this_fold = gnn_fold[gnn_fold["fold"] == fold["name"]].set_index("txId")["gnn_score"]
    gnn_fold_scores[fold["name"]] = this_fold.loc[va["txId"]].values
    print(fold["name"], "GNN AUC:", round(roc_auc_score(va["label"], gnn_fold_scores[fold["name"]]), 4))

fold 1 GNN AUC: 0.9615
fold 2 GNN AUC: 0.9122
fold 3 GNN AUC: 0.9857


## 6. The blend: choosing the GNN weight

`blend = (1 - w) x LightGBM rank + w x GNN rank`. The table scores each weight on the folds.

In [10]:
GNN_WEIGHTS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]

rows = []
for w in GNN_WEIGHTS:
    row = {"gnn_weight": w}
    for fold in FOLDS:
        tr, va = split_fold(labeled, fold)
        blend = (1 - w) * to_rank(lgbm_fold_scores[fold["name"]]) + w * to_rank(gnn_fold_scores[fold["name"]])
        row[fold["name"]] = roc_auc_score(va["label"], blend)
    rows.append(row)

blend_table = pd.DataFrame(rows).set_index("gnn_weight")
blend_table["mean"] = blend_table[fold_names].mean(axis=1)
blend_table.round(4)

,fold 1,fold 2,fold 3,mean
gnn_weight,,,,
0.0,0.9888,0.9572,0.9965,0.9808
0.1,0.9900,0.9548,0.9978,0.9809
0.2,0.9906,0.9517,0.9983,0.9802
0.3,0.9903,0.9479,0.9983,0.9788
0.4,0.9892,0.9437,0.9980,0.9770
0.5,0.9872,0.9392,0.9974,0.9746
1.0,0.9615,0.9122,0.9857,0.9531


**Why the final weight is 0.4:** The folds and the public leaderboard disagree about the GNN:

| GNN weight | 0 | 0.1 | 0.2 | 0.3 | **0.4** | 0.5 | 0.6 | 0.7 |
|---|---|---|---|---|---|---|---|---|
| Public leaderboard | 0.94660 | 0.95037* | 0.95369 | 0.95581 | **0.95654** | 0.95604 | 0.95427 | 0.95104 |

\* the 0.1 submission used a single GNN; the others use the average of five.

- On the **folds**, small weights are level with LightGBM alone and larger weights lower the mean,
  almost entirely through fold 2 (step 24), where the GNN does badly on the new kind of illicit
  transaction.
- On the **public leaderboard**, the score rises steadily up to 0.4 and falls after it: a smooth curve
  across eight submissions, not a single lucky result.

I chose 0.4 because the public leaderboard is scored on the same time period as the final (private)
leaderboard, while the folds come from earlier periods that have already behaved differently from the
test period. The risk is the one the folds point to: if the private test rows contain more of the new
behaviour seen at step 24, the GNN could hurt. That is why my second selected submission is LightGBM
alone (`sub01_baseline`): Kaggle uses the better of the two selected submissions, so one of them should
be well placed either way.

## 7. Final models and submission

LightGBM is retrained on all labeled rows (steps 1 to 35) so that it sees the steps closest to the test
period; the GNN script already did the same for the GNN. The two test rankings are blended with
weight 0.4 on the GNN.

In [11]:
GNN_WEIGHT = 0.4

final_lgbm = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
final_lgbm.fit(labeled[raw_cols], labeled["label"])
lgbm_test_rank = to_rank(final_lgbm.predict_proba(test[raw_cols])[:, 1])

# GNN predictions in test's row order
gnn_test_rank = to_rank(gnn_test.set_index("txId")["gnn_score"].loc[test["txId"]].values)

test_pred = (1 - GNN_WEIGHT) * lgbm_test_rank + GNN_WEIGHT * gnn_test_rank
submission = pd.DataFrame({"index": test["index"], "target": test_pred})

# Format checks
assert len(submission) == 15329, "wrong number of rows"
assert list(submission.columns) == ["index", "target"], "wrong columns"
assert submission["target"].notna().all(), "NaN in predictions"
assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

SUB_PATH = f"{SUB_DIR}/e0968110_final_submission.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved", SUB_PATH)
submission.head()

Saved submissions/e0968110_final_submission.csv


,index,target
0,0,0.647974
1,1,0.744223
2,2,0.581590
3,3,0.592341
4,4,0.618331


In [ ]:
# Reproducibility check
SUBMITTED = f"{SUB_DIR}/sub10_gnn5_blend_w04.csv"
if os.path.exists(SUBMITTED):
    submitted = pd.read_csv(SUBMITTED)
    same = np.allclose(submitted["target"].values, submission["target"].values)
    print("Matches", SUBMITTED, ":", same)
else:
    print(SUBMITTED, "not found; skipping the comparison")

Matches submissions/sub10_gnn5_blend_w04.csv : True


## 8. Summary

- **Final model:** LightGBM (default settings, raw features) blended with an average of five directed
  GraphSAGE networks, GNN weight 0.4. Public leaderboard 0.95654, against 0.94660 for LightGBM alone and
  0.92119 for the course baseline.
- **Final selections:** this blend (`sub10_gnn5_blend_w04`) and LightGBM alone (`sub01_baseline`).
- **Main lesson:** the hardest part of this problem is change over time. The illicit transactions that
  were hardest to catch were a new pattern of connected illicit transactions that hardly appears in
  earlier steps. Most changes that helped on familiar periods hurt on that one, and the validation folds
  and the public leaderboard sometimes disagreed, so each choice had to weigh both.